# W1D4: 밝기·대비 조절

> 📖 원리 이해: [LearnOpenCV - Histogram Equalization](https://learnopencv.com/histogram-equalization/)
> 📖 원리 이해: [LearnOpenCV - CLAHE: Contrast Limited Adaptive Histogram Equalization](https://learnopencv.com/clahe-histogram-equalization-opencv/)
> 📋 파라미터 확인: [OpenCV 공식 - equalizeHist](https://docs.opencv.org/4.x/d6/dc7/group__imgproc__hist.html#ga7e54091f0c937d49bf84152a16f76d6e)
> 📋 파라미터 확인: [OpenCV 공식 - createCLAHE](https://docs.opencv.org/4.x/d6/dc7/group__imgproc__hist.html#ga7e54091f0c937d49bf84152a16f76d6e)

---

## 🧠 원리 이해

### equalizeHist — 왜 CDF로 대비를 높일 수 있나?

W1D3에서 배운 **CDF(누적 히스토그램)** 를 그대로 활용합니다.

비유: 안경 없이 찍은 사진처럼 밝기가 좁은 구간에 몰려 뿌옇게 보이는 이미지를, 안경을 끼고 본 것처럼 0~255 전체로 펼쳐주는 작업입니다.

**3단계 동작 과정:**

1. **히스토그램 계산** — 각 밝기값(0~255)의 픽셀 수를 셈
2. **CDF 계산** — `cdf[i]` = 밝기 0부터 i까지의 픽셀 수 누적합
3. **픽셀값 재매핑** — `새 밝기 = round( cdf[원래 밝기] / 전체 픽셀 수 × 255 )`

```
예시: 전체 픽셀의 80%가 밝기 100~150에 집중된 이미지

  cdf[150] ≈ 전체의 90%  →  새 밝기 = round(0.90 × 255) = 230
  cdf[100] ≈ 전체의 10%  →  새 밝기 = round(0.10 × 255) =  26

결과: 원래 100~150이던 픽셀들이 26~230으로 펼쳐짐 → 대비 상승
```

결과적으로 **CDF가 직선에 가까워**지고, 히스토그램은 0~255에 고르게 분포됩니다.

**한계**: 이미지 전체를 하나의 히스토그램으로 처리하므로, 조명이 불균일한 이미지(한쪽은 밝고 다른쪽은 어두움)에서는 밝은 영역이 과포화(흰색으로 날아감)되고 어두운 영역의 노이즈가 과장됩니다.

---

### CLAHE — equalizeHist의 두 한계를 각각 해결

**CLAHE = Contrast Limited Adaptive Histogram Equalization**

| equalizeHist 문제 | CLAHE 해결책 | 키워드 |
|-------------------|--------------|--------|
| 전체 이미지를 한 번에 처리 → 조명 편차에 취약 | 이미지를 N×M 타일로 분할해 각 타일에 독립 equalizeHist 적용 | **Adaptive** |
| 특정 밝기에 픽셀이 몰리면 그 밝기가 과도하게 강조 → 노이즈 증폭 | 히스토그램의 bin 높이가 `clipLimit` 초과 시 잘라내고 나머지 bin에 균등 재분배 | **Contrast Limited** |

**clipLimit 동작 상세:**

```
히스토그램에서 bin 높이 > clipLimit 이면:
  1. 초과분을 잘라냄 (clip)
  2. 잘린 픽셀 수를 모든 bin에 균등하게 더함
  3. 재분배된 히스토그램으로 equalizeHist 적용
```

- `clipLimit` 낮을수록 → 더 많이 잘림 → 대비 강화 억제 → 노이즈 작음
- `clipLimit` 무한히 크면 → 아무것도 안 잘림 → equalizeHist와 동일
- 실무 권장 시작값: `clipLimit=2.0`

**타일 경계 처리:**

각 타일을 독립 평활화하면 타일 경계가 이음새처럼 보일 수 있습니다. CLAHE는 인접한 4개 타일의 변환 결과를 **양선형 보간(bilinear interpolation)** 으로 섞어 경계를 자연스럽게 연결합니다.

```
tileGridSize=(8,8)  →  이미지를 8×8 = 64개 타일로 분할
각 타일에 clipLimit 적용 후 equalizeHist
타일 경계는 양선형 보간으로 부드럽게 연결
```

---

**한 줄 요약**: `equalizeHist` = 전체 이미지 CDF를 직선화 / `CLAHE` = 타일로 쪼갠 equalizeHist + clipLimit으로 노이즈 억제

---

## 📌 오늘의 목표
- 히스토그램 평활화로 대비가 낮은 이미지를 개선하기
- `equalizeHist`의 원리(CDF → 평탄화)를 이해하기
- `equalizeHist`의 한계와 `createCLAHE`가 왜 필요한지 파악하기
- 6종 결함에 CLAHE를 적용해 검사 가시성 향상 확인하기

## 🎯 배울 함수

### `cv2.equalizeHist(src)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `src` | `uint8 grayscale` | **반드시 8비트 그레이스케일** 이미지. BGR 입력 시 오류 |
| 반환값 | `uint8 array` | 히스토그램이 평탄화된 이미지. 입력과 동일한 shape |

> 원리: W1D3에서 배운 CDF를 0~255 전체에 고르게 펼쳐, 대비를 자동으로 최대화합니다.

---

### `cv2.createCLAHE(clipLimit, tileGridSize)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `clipLimit` | `float` | 히스토그램 클리핑 한계 (기본값 `40.0`). 낮을수록 보수적 대비 강화, 높을수록 equalizeHist에 가까워짐 |
| `tileGridSize` | `(int, int)` | 타일 격자 크기 (기본값 `(8, 8)`). 이미지를 N×M 타일로 나눠 각 타일에 독립 평활화 적용 |
| 반환값 | `CLAHE 객체` | `.apply(img)`로 이미지에 적용 |

> `equalizeHist` vs `CLAHE`: equalizeHist는 이미지 전체를 한 번에 평활화 → 밝은 영역이 날아가거나 노이즈가 과장됨. CLAHE는 작은 타일마다 독립 적용 + clipLimit으로 과도한 대비 억제 → 디테일 보존.

## 📦 import + 데이터 로딩

대비 조절 실험에는 어두운 영역과 밝은 영역이 공존하는 이미지가 효과를 잘 보여줍니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path("../data/kaggle/archive/NEU-DET/train/images")
DEFECT_TYPES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

img = cv2.imread(str(DATA_DIR / "inclusion" / "inclusion_1.jpg"), cv2.IMREAD_GRAYSCALE)
print(f"로딩 완료: shape={img.shape}, dtype={img.dtype}")
print(f"픽셀 범위: {img.min()} ~ {img.max()}, 평균: {img.mean():.1f}, 표준편차: {img.std():.1f}")

## 🔧 1. equalizeHist — 히스토그램 평활화

**`cv2.equalizeHist(img)`** 는 CDF를 이용해 히스토그램을 0~255 전체에 고르게 펼칩니다.

- **왜 CDF를 쓰나?** W1D3에서 본 것처럼 CDF는 "밝기 X 이하 픽셀 비율"을 나타냅니다. CDF 값에 255를 곱하면 새 밝기값이 되므로, 원래 좁은 범위에 몰려 있던 픽셀들이 0~255 전체에 펼쳐집니다.
- **입력 조건**: 반드시 **8비트 그레이스케일**. BGR 이미지를 그대로 넣으면 오류 발생.
- **제조 검사 연결**: 조명 조건이 불안정한 라인(야간/주간 밝기 차이)에서 촬영한 이미지를 정규화할 때 사용합니다. 다만 전체 이미지에 동일하게 적용하므로 조명이 균일하지 않은 경우 부작용이 생길 수 있습니다.

In [ ]:
img_eq = cv2.equalizeHist(img)

hist_orig = cv2.calcHist([img],    [0], None, [256], [0, 256])
hist_eq   = cv2.calcHist([img_eq], [0], None, [256], [0, 256])

print(f"원본   픽셀 범위: {img.min():3d} ~ {img.max():3d}, 평균: {img.mean():.1f}, 표준편차: {img.std():.1f}")
print(f"평활화 픽셀 범위: {img_eq.min():3d} ~ {img_eq.max():3d}, 평균: {img_eq.mean():.1f}, 표준편차: {img_eq.std():.1f}")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title(f"원본  (std={img.std():.1f})", fontsize=13)
axes[0, 0].axis('off')

axes[0, 1].imshow(img_eq, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title(f"equalizeHist  (std={img_eq.std():.1f})", fontsize=13)
axes[0, 1].axis('off')

axes[1, 0].plot(hist_orig, color='steelblue', linewidth=1.5)
axes[1, 0].fill_between(range(256), hist_orig.flatten(), alpha=0.3, color='steelblue')
axes[1, 0].set_title("원본 히스토그램", fontsize=13)
axes[1, 0].set_xlabel("밝기 (0~255)", fontsize=11)
axes[1, 0].set_ylabel("픽셀 수", fontsize=11)
axes[1, 0].set_xlim([0, 256])
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(hist_eq, color='darkorange', linewidth=1.5)
axes[1, 1].fill_between(range(256), hist_eq.flatten(), alpha=0.3, color='darkorange')
axes[1, 1].set_title("equalizeHist 후 히스토그램", fontsize=13)
axes[1, 1].set_xlabel("밝기 (0~255)", fontsize=11)
axes[1, 1].set_ylabel("픽셀 수", fontsize=11)
axes[1, 1].set_xlim([0, 256])
axes[1, 1].grid(alpha=0.3)

plt.suptitle("equalizeHist 전/후 비교 (inclusion_1)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **표준편차(std)** 가 높아졌다면 = 밝기 분포가 넓어졌다 = 대비 향상됨
- 평활화 후 히스토그램이 **빈 구간(0인 bin)** 이 많이 생기는 것은 정상. 원래 붙어있던 밝기값이 펼쳐지면서 사이사이에 공백이 생깁니다.
- **결함 부위**: 원본에서 어둡게 뭉쳐있던 결함이 평활화 후 더 뚜렷하게 나타나는지 확인
- 이미지 전체 밝기가 고르게 펼쳐지므로 전반적으로 대비가 높아짐

> 🤔 **판단 질문:** 평활화 전 히스토그램에서 봉우리가 한쪽에 치우쳐 있었다면, 평활화 후 이미지는 밝아졌을까요 어두워졌을까요?

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- `img` 대신 BGR로 로딩한 이미지(`cv2.IMREAD_COLOR`)를 `equalizeHist`에 넣으면 어떻게 될까요? 오류가 날까요, 이상한 결과가 나올까요?
- `patches_1.jpg`로 바꿔서 비교해보세요. inclusion보다 평활화 효과가 더 클까요 작을까요?
- 히스토그램이 이미 넓게 퍼져있는 이미지에 `equalizeHist`를 적용하면 변화가 거의 없을까요?

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 2. equalizeHist의 한계 — 전체 평활화의 부작용

`equalizeHist`는 이미지 **전체**를 하나의 히스토그램으로 봅니다.
- **문제**: 조명이 불균일한 이미지(한쪽은 밝고 다른쪽은 어두움)에서는 밝은 쪽이 과포화(날아감)되고, 어두운 쪽은 노이즈가 과장됩니다.
- 결함보다 조명 차이가 더 강조되어 오히려 검출이 어려워질 수 있습니다.

**제조 검사 연결**: 실제 라인에서 조명 편차가 있는 경우 equalizeHist를 쓰면 역효과가 납니다. 이 한계를 극복한 것이 CLAHE입니다.

In [ ]:
img_crazing = cv2.imread(str(DATA_DIR / "crazing" / "crazing_1.jpg"), cv2.IMREAD_GRAYSCALE)
img_rolled  = cv2.imread(str(DATA_DIR / "rolled-in_scale" / "rolled-in_scale_1.jpg"), cv2.IMREAD_GRAYSCALE)

samples = [
    ("crazing_1",         img_crazing),
    ("rolled-in_scale_1", img_rolled),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 11))

for row, (name, src) in enumerate(samples):
    eq = cv2.equalizeHist(src)

    hist_s = cv2.calcHist([src], [0], None, [256], [0, 256])
    hist_e = cv2.calcHist([eq],  [0], None, [256], [0, 256])

    axes[row, 0].imshow(src, cmap='gray', vmin=0, vmax=255)
    axes[row, 0].set_title(f"{name}\n원본 (std={src.std():.1f})", fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(eq, cmap='gray', vmin=0, vmax=255)
    axes[row, 1].set_title(f"equalizeHist (std={eq.std():.1f})", fontsize=11)
    axes[row, 1].axis('off')

    axes[row, 2].plot(hist_s, color='steelblue', linewidth=1.5)
    axes[row, 2].fill_between(range(256), hist_s.flatten(), alpha=0.3, color='steelblue')
    axes[row, 2].set_title("원본 히스토그램", fontsize=11)
    axes[row, 2].set_xlim([0, 256])
    axes[row, 2].grid(alpha=0.3)

    axes[row, 3].plot(hist_e, color='darkorange', linewidth=1.5)
    axes[row, 3].fill_between(range(256), hist_e.flatten(), alpha=0.3, color='darkorange')
    axes[row, 3].set_title("평활화 히스토그램", fontsize=11)
    axes[row, 3].set_xlim([0, 256])
    axes[row, 3].grid(alpha=0.3)

plt.suptitle("equalizeHist: 결함 유형별 부작용 확인", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- 결함이 더 선명해졌는지, 아니면 노이즈나 배경 텍스처도 함께 강조됐는지 주목
- **과포화(하이라이트 날아감)**: 밝은 영역 전체가 흰색으로 뭉개지는 현상 → 결함 정보 손실 가능
- **노이즈 과장**: 어두운 영역에서 원래 보이지 않던 점들이 두드러지는 현상
- std가 많이 올랐다고 항상 좋은 건 아닙니다 — **검출에 유용한 대비**인지 확인 필요

> 🤔 **판단 질문:** 어떤 결함 유형에서 equalizeHist 부작용이 가장 심하게 보이나요? 그 이유는 원본 히스토그램 모양과 어떤 관련이 있을까요?

## 🔧 3. createCLAHE — 적응형 히스토그램 평활화

**CLAHE (Contrast Limited Adaptive Histogram Equalization)** 는 equalizeHist의 두 가지 한계를 해결합니다:

1. **Adaptive**: 이미지를 작은 타일(격자)로 나눠 각 타일마다 독립적으로 평활화 → 조명 불균일 문제 해결
2. **Contrast Limited**: 각 타일의 히스토그램에서 `clipLimit`을 초과하는 부분을 잘라내고 재분배 → 노이즈 과장 억제

```python
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
```

**제조 검사 연결**: AOI에서 넓은 PCB나 강판 전체를 균일하게 검사할 때 CLAHE가 표준 전처리로 쓰입니다. 조명 핫스팟(한 부분만 특히 밝음)이 있어도 타일 단위 처리로 전체를 고르게 만듭니다.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_eq    = cv2.equalizeHist(img)
img_clahe = clahe.apply(img)

hist_orig  = cv2.calcHist([img],       [0], None, [256], [0, 256])
hist_eq    = cv2.calcHist([img_eq],    [0], None, [256], [0, 256])
hist_clahe = cv2.calcHist([img_clahe], [0], None, [256], [0, 256])

print(f"원본       픽셀 범위: {img.min():3d} ~ {img.max():3d}, std={img.std():.1f}")
print(f"equalizeHist 범위: {img_eq.min():3d} ~ {img_eq.max():3d}, std={img_eq.std():.1f}")
print(f"CLAHE      범위: {img_clahe.min():3d} ~ {img_clahe.max():3d}, std={img_clahe.std():.1f}")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

titles = ["원본", "equalizeHist", "CLAHE (clip=2.0, tile=8×8)"]
images = [img, img_eq, img_clahe]
hists  = [hist_orig, hist_eq, hist_clahe]
colors = ['steelblue', 'darkorange', 'seagreen']

for col, (title, im, h, c) in enumerate(zip(titles, images, hists, colors)):
    axes[0, col].imshow(im, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f"{title}\nstd={im.std():.1f}", fontsize=12)
    axes[0, col].axis('off')

    axes[1, col].plot(h, color=c, linewidth=1.5)
    axes[1, col].fill_between(range(256), h.flatten(), alpha=0.3, color=c)
    axes[1, col].set_title(f"{title} 히스토그램", fontsize=12)
    axes[1, col].set_xlabel("밝기 (0~255)", fontsize=11)
    axes[1, col].set_ylabel("픽셀 수", fontsize=11)
    axes[1, col].set_xlim([0, 256])
    axes[1, col].grid(alpha=0.3)

plt.suptitle("원본 vs equalizeHist vs CLAHE 비교 (inclusion_1)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 📊 SNR 정량 평가 — 원본 vs equalizeHist vs CLAHE

**SNR (Signal-to-Noise Ratio)** 로 어떤 처리가 결함을 배경 노이즈 대비 더 잘 드러내는지 수치로 비교합니다.

```
SNR = 결함 영역 평균밝기 / 배경 영역 표준편차
```

- **분자(결함 평균)**: 결함 신호의 강도
- **분모(배경 std)**: 배경의 노이즈 정도
- **SNR 높을수록** → 결함이 노이즈 대비 잘 구분됨 → 좋은 전처리

> ⚠️ ROI 좌표는 이미지마다 다릅니다. 아래 시각화 셀에서 결함 위치를 확인 후 `DEFECT_ROI`, `BG_ROI`를 조정하세요.

In [ ]:
# ROI 좌표 설정 (y1:y2, x1:x2) — 결과 보고 조정하세요
DEFECT_ROI = (80, 190, 120, 140)   # 결함 있는 영역
BG_ROI     = (100, 150,  20,  70)  # 결함 없는 배경 영역

dy1, dy2, dx1, dx2 = DEFECT_ROI
by1, by2, bx1, bx2 = BG_ROI

# ROI 위치 시각화
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(img, cmap='gray', vmin=0, vmax=255)
ax.add_patch(plt.Rectangle((dx1, dy1), dx2-dx1, dy2-dy1,
             edgecolor='red',   facecolor='none', linewidth=2, label='결함 ROI'))
ax.add_patch(plt.Rectangle((bx1, by1), bx2-bx1, by2-by1,
             edgecolor='green', facecolor='none', linewidth=2, label='배경 ROI'))
ax.legend(fontsize=11)
ax.set_title("ROI 위치 확인 (빨강=결함, 초록=배경)\n좌표가 맞지 않으면 DEFECT_ROI, BG_ROI 수정", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

# SNR 계산 (대비 기반)
def calc_snr(image, defect_roi, bg_roi):
    dy1, dy2, dx1, dx2 = defect_roi
    by1, by2, bx1, bx2 = bg_roi
    defect_mean = image[dy1:dy2, dx1:dx2].mean()
    bg_mean     = image[by1:by2, bx1:bx2].mean()
    bg_std      = image[by1:by2, bx1:bx2].std()
    contrast    = abs(defect_mean - bg_mean)
    return contrast / (bg_std + 1e-6)

print("=" * 55)
print(f"{'':12s}  {'결함 평균':>8s}  {'배경 평균':>8s}  {'배경 std':>8s}  {'SNR':>6s}")
print("-" * 55)
for name, im in [("원본", img), ("equalizeHist", img_eq), ("CLAHE", img_clahe)]:
    dy1, dy2, dx1, dx2 = DEFECT_ROI
    by1, by2, bx1, bx2 = BG_ROI
    d_mean   = im[dy1:dy2, dx1:dx2].mean()
    bg_mean  = im[by1:by2, bx1:bx2].mean()
    bg_std   = im[by1:by2, bx1:bx2].std()
    contrast = abs(d_mean - bg_mean)
    snr      = contrast / (bg_std + 1e-6)
    print(f"{name:12s}  {d_mean:8.1f}  {bg_mean:8.1f}  {bg_std:8.1f}  {snr:6.2f}")
print("=" * 55)
print("\n📊 SNR = |결함평균 - 배경평균| / 배경std")
print("   → 결함이 밝든 어둡든 배경과의 대비를 노이즈로 나눈 값")

### 해석 가이드
- **equalizeHist**: 히스토그램이 매우 넓게 펼쳐짐 → std 최대 → 하지만 결함 외 노이즈도 강조
- **CLAHE**: equalizeHist보다 std가 적당히 낮음 → 과도한 대비 억제 → 결함 디테일 보존
- **결함 주변**: CLAHE 결과에서 결함 경계가 equalizeHist보다 자연스럽게 강조됐는지 확인
- CLAHE 히스토그램은 원본과 equalizeHist의 중간 형태 — 평탄하지만 급격하지 않음

> 🤔 **판단 질문:** CLAHE 결과에서 결함 주변의 텍스처가 equalizeHist보다 더 자세히 보인다면, 그 이유는 clipLimit 때문일까요 tileGridSize 때문일까요?

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- `clipLimit=40.0`으로 바꾸면 CLAHE 결과가 equalizeHist와 비슷해질까요? (clipLimit이 무한히 크면 어떻게 될까요?)
- `tileGridSize=(1, 1)`로 바꾸면 어떻게 될까요? (타일이 1개 = 전체 이미지 한 번에 처리)
- `tileGridSize=(64, 64)`로 매우 작은 타일을 쓰면 어떤 부작용이 생길까요?

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 4. CLAHE 파라미터 실험 — clipLimit vs tileGridSize

CLAHE의 두 파라미터는 서로 다른 역할을 합니다:
- **`clipLimit`**: 대비 강화 강도 조절. 낮을수록(≈1.0) 원본에 가깝고, 높을수록(≈40) equalizeHist에 가까움
- **`tileGridSize`**: 타일 크기 조절. 작은 타일(큰 값)일수록 더 지역적 적응, 크기가 클수록 더 전역적

**제조 검사 연결**: 검사 카메라의 FOV(Field of View) 크기와 결함 크기에 맞게 `tileGridSize`를 설정합니다. 결함이 작고 촘촘하면 작은 타일이 유리하지만, 너무 작으면 타일 경계가 이음새처럼 보입니다.

In [ ]:
img_test = cv2.imread(str(DATA_DIR / "pitted_surface" / "pitted_surface_1.jpg"), cv2.IMREAD_GRAYSCALE)

clip_limits = [1.0, 2.0, 5.0, 15.0]
tile_sizes  = [(4, 4), (8, 8), (16, 16), (32, 32)]

fig, axes = plt.subplots(2, 4, figsize=(22, 12))

for col, clip in enumerate(clip_limits):
    c = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    result = c.apply(img_test)
    axes[0, col].imshow(result, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f"clipLimit={clip}\ntile=(8×8), std={result.std():.1f}", fontsize=11)
    axes[0, col].axis('off')

for col, tile in enumerate(tile_sizes):
    c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=tile)
    result = c.apply(img_test)
    axes[1, col].imshow(result, cmap='gray', vmin=0, vmax=255)
    axes[1, col].set_title(f"tile={tile[0]}×{tile[1]}\nclipLimit=2.0, std={result.std():.1f}", fontsize=11)
    axes[1, col].axis('off')

fig.text(0.01, 0.75, 'clipLimit\n변화', ha='left', va='center', fontsize=13, fontweight='bold',
         rotation=0, color='steelblue')
fig.text(0.01, 0.28, 'tileGridSize\n변화', ha='left', va='center', fontsize=13, fontweight='bold',
         rotation=0, color='seagreen')

plt.suptitle("CLAHE 파라미터 실험 (pitted_surface_1)", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.show()

### 해석 가이드
- **clipLimit 증가**: 대비가 강해짐, 특정 값 이상부터는 노이즈가 눈에 띄게 증가
- **tileGridSize 작게 (4×4)**: 타일이 많아 지역 적응이 강함 → 작은 결함에 유리하지만 타일 경계 인공물 가능
- **tileGridSize 크게 (32×32)**: 타일이 적어 거의 전역 평활화에 가까워짐 → 부드럽지만 지역 대비 약화
- **검사 최적값 탐색 방법**: 결함이 가장 선명하게 보이면서 배경 노이즈는 최소인 조합 선택

> 🤔 **판단 질문:** pitted_surface(표면 움푹 파임)에서 결함 검출에 가장 적합한 clipLimit과 tileGridSize 조합은 어떤 것인가요? 판단 근거는?

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- `clipLimit=1.0, tileGridSize=(4,4)` 조합과 `clipLimit=5.0, tileGridSize=(32,32)` 조합 중 pitted_surface 결함 검출에 어느 쪽이 유리한지 예측해보세요.
- 원본 이미지의 std가 이미 높은(예: scratches) 결함에 CLAHE를 적용하면 시각적 개선이 클까요 작을까요?
- `tileGridSize=(200, 200)`처럼 이미지 크기보다 큰 타일을 지정하면 어떤 일이 생길까요?

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 5. 6종 결함에 equalizeHist vs CLAHE 적용

결함마다 원본의 밝기 분포가 다르므로, 대비 향상의 효과도 다르게 나타납니다.

**제조 검사 연결**: 다음 주부터 배울 threshold(임계값 분리)를 적용하기 전에 CLAHE로 전처리하면 결함 검출 정확도가 크게 높아집니다. 전처리 → 분리 → 검출의 파이프라인 첫 단계입니다.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

fig, axes = plt.subplots(3, 6, figsize=(26, 13))

row_labels = ["원본", "equalizeHist", "CLAHE\n(clip=2.0, tile=8×8)"]
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=12, fontweight='bold', rotation=90, labelpad=10)

for col, (defect, color) in enumerate(zip(DEFECT_TYPES, colors)):
    src = cv2.imread(str(DATA_DIR / defect / f"{defect}_1.jpg"), cv2.IMREAD_GRAYSCALE)
    eq  = cv2.equalizeHist(src)
    cl  = clahe.apply(src)

    short_name = defect.replace('_', '\n')

    axes[0, col].imshow(src, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f"{short_name}\nstd={src.std():.1f}", fontsize=10, color=color, fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].imshow(eq, cmap='gray', vmin=0, vmax=255)
    axes[1, col].set_title(f"std={eq.std():.1f}", fontsize=10)
    axes[1, col].axis('off')

    axes[2, col].imshow(cl, cmap='gray', vmin=0, vmax=255)
    axes[2, col].set_title(f"std={cl.std():.1f}", fontsize=10)
    axes[2, col].axis('off')

plt.suptitle("6종 결함: 원본 vs equalizeHist vs CLAHE", fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **std 변화**: 원본 → equalizeHist → CLAHE 순서로 std가 어떻게 변하는지 확인
- **결함 가시성**: 세 버전 중 어느 것이 결함을 가장 명확히 보여주는지 각 유형별로 판단
- **crazing**: 미세 균열이 CLAHE에서 더 선명하게 나타나는지 확인
- **inclusion/patches**: 어두운 결함 부위가 equalizeHist에서 날아갔는지 CLAHE에서는 보존됐는지 비교
- **scratches**: 선형 결함은 어떤 처리에서 가장 잘 보이나요?

> 🤔 **판단 질문:** 결함 유형 6개 중 CLAHE보다 equalizeHist가 더 나은 결과를 보이는 경우가 있나요? 있다면 그 이유는?

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- 각 결함 유형에서 `"_1.jpg"`를 `"_10.jpg"`로 바꿔 같은 결함의 다른 이미지에도 같은 패턴이 나오는지 확인해보세요.
- CLAHE `clipLimit=0.5`로 낮추면 어떤 결함에서 효과가 거의 사라지나요?
- std가 가장 낮은 결함 유형을 찾아보고, 그 유형에서 CLAHE 효과가 가장 큰지 확인해보세요.

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔬 파라미터 실험: clipLimit 수치와 결함 가시성

검사 파이프라인에서 CLAHE의 `clipLimit`을 어떻게 선택할지, 정량적으로 비교합니다.
- **std**: 대비 정도 측정
- 실무에서는 이 이후 단계(threshold 후 결함 면적)로 최적값을 결정하지만, 여기서는 std로 대리 지표를 사용합니다.

In [ ]:
clip_values = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0, 15.0, 25.0, 40.0]
defect_samples = ["crazing", "inclusion", "scratches"]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors_d = ['#e74c3c', '#3498db', '#9b59b6']

for defect, color in zip(defect_samples, colors_d):
    src = cv2.imread(str(DATA_DIR / defect / f"{defect}_1.jpg"), cv2.IMREAD_GRAYSCALE)
    std_vals = []
    for clip in clip_values:
        c = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
        result = c.apply(src)
        std_vals.append(result.std())
    axes[0].plot(clip_values, std_vals, marker='o', color=color, linewidth=2, label=defect)

eq_std = {d: cv2.equalizeHist(
    cv2.imread(str(DATA_DIR / d / f"{d}_1.jpg"), cv2.IMREAD_GRAYSCALE)
).std() for d in defect_samples}

for defect, color in zip(defect_samples, colors_d):
    axes[0].axhline(y=eq_std[defect], color=color, linestyle='--', alpha=0.5,
                    label=f"{defect} equalizeHist")

axes[0].set_title("clipLimit vs std (tileGridSize=8×8)", fontsize=13)
axes[0].set_xlabel("clipLimit", fontsize=11)
axes[0].set_ylabel("표준편차 (std)", fontsize=11)
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(alpha=0.3)
axes[0].set_xscale('log')

tile_sizes_n = [2, 4, 8, 16, 32]
for defect, color in zip(defect_samples, colors_d):
    src = cv2.imread(str(DATA_DIR / defect / f"{defect}_1.jpg"), cv2.IMREAD_GRAYSCALE)
    std_vals = []
    for t in tile_sizes_n:
        c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(t, t))
        result = c.apply(src)
        std_vals.append(result.std())
    axes[1].plot(tile_sizes_n, std_vals, marker='s', color=color, linewidth=2, label=defect)

axes[1].set_title("tileGridSize vs std (clipLimit=2.0)", fontsize=13)
axes[1].set_xlabel("tileGridSize (N×N)", fontsize=11)
axes[1].set_ylabel("표준편차 (std)", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_xticks(tile_sizes_n)
axes[1].set_xticklabels([f"{t}×{t}" for t in tile_sizes_n])

plt.suptitle("CLAHE 파라미터별 대비 강도 변화", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 clipLimit 2.0, tileGridSize 8×8 기준 std 비교:")
for defect in defect_samples:
    src = cv2.imread(str(DATA_DIR / defect / f"{defect}_1.jpg"), cv2.IMREAD_GRAYSCALE)
    c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    result = c.apply(src)
    print(f"  {defect:20s}: 원본 std={src.std():.1f}  →  CLAHE std={result.std():.1f}  (증가: +{result.std()-src.std():.1f})")

### 해석 가이드
- **clipLimit 그래프**: 특정 값 이상에서 std 증가가 완만해지는 구간이 equalizeHist의 std와 가까워지는 지점
- **결함별 차이**: 원본 std가 낮은 결함일수록 CLAHE 적용 후 std 증가폭이 큼
- **tileGridSize 그래프**: 타일이 작아질수록(숫자 커질수록) std가 높아지는 경향 → 지역 적응이 강해짐
- **실무 권장**: 결함 유형에 맞게 clipLimit=1.5~3.0, tileGridSize=(8,8)이 일반적 출발점

> 🤔 **판단 질문:** clipLimit을 10배로 높였을 때 std가 2배가 되지 않는 이유는 뭘까요?

## 📝 정리

### 오늘 배운 것
| 함수/개념 | 핵심 포인트 |
|-----------|-------------|
| `cv2.equalizeHist(img)` | CDF 기반 전체 히스토그램 평활화. **입력은 반드시 uint8 그레이스케일** |
| `cv2.createCLAHE(clipLimit, tileGridSize)` | 타일 단위 적응형 평활화. clipLimit으로 과도한 대비 억제 |
| `clahe.apply(img)` | CLAHE 객체를 이미지에 적용 |
| `clipLimit` | 낮을수록 보수적(원본에 가깝), 높을수록 equalizeHist에 가까워짐 |
| `tileGridSize` | 작은 타일(큰 N) = 강한 지역 적응, 큰 타일(작은 N) = 약한 지역 적응 |

### 핵심 개념
- `equalizeHist` = W1D3 CDF를 직접 사용해 히스토그램을 평탄화. CDF가 직선이 되도록 픽셀값 재배치
- **전체 vs 지역**: equalizeHist는 전역 평활화 → 조명 불균일 이미지에서 부작용. CLAHE는 타일별 독립 적용 → 부작용 억제
- **검사 파이프라인**: `imread → (CLAHE 전처리) → threshold → contour 검출` 순서가 다음 주 목표
- **std는 대리 지표**: 대비 강도를 수치화하는 편리한 방법. 높다고 항상 좋은 게 아니라 결함 선명도가 기준

### 복습 퀴즈
1. `equalizeHist`에 BGR 컬러 이미지를 넣으면 어떤 일이 생기나요? 올바른 방법은?
2. `clipLimit`을 매우 크게(예: 1000) 설정하면 CLAHE는 어떤 함수와 동일하게 동작하나요? 왜?
3. `tileGridSize=(1,1)`로 설정한 CLAHE와 `equalizeHist`는 결과가 같을까요? 왜 같거나 다를까요?
4. 원본 이미지의 히스토그램이 이미 0~255 전체에 고르게 분포되어 있다면, equalizeHist 적용 전후 차이가 거의 없는 이유는?
5. 제조 검사에서 조명 편차가 심한 이미지에 equalizeHist 대신 CLAHE를 써야 하는 이유를 한 문장으로 설명하세요.

### 다음에 할 것 (W1D5)
- 이미지 연산 — `cv2.absdiff`, `cv2.addWeighted`, `cv2.bitwise_and/or/not`
- 두 이미지 간 차이를 계산해 변화 영역 감지하기 (결함 vs 정상 비교의 기초)